<a href="https://colab.research.google.com/github/louistrue/DB-1/blob/main/DB1_W10_Mengen_Auswertungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DB1 – Woche 10: Mengen, Attribute und Auswertungen
**Velobrücke Aaretal – Brücke 2 (erweitert)**

---

In Woche 8 habt ihr ein Modell **automatisch geprüft**: Stimmt die Geometrie? Sind alle Knoten verbunden? Heute baut ihr darauf auf.

Das Modell hat **neue Attribute** bekommen: `material` und `querschnittsflaeche_m2`. Damit könnt ihr nicht mehr nur prüfen, sondern auch **auswerten**:

1. **Modellprüfung erweitert**: Sind die neuen Attribute korrekt? *(rot/grün Visualisierung)*
2. **Mengen ermitteln**: Wie viel Stahl, wie viel Holz steckt in der Brücke?
3. **Auswertungen visualisieren**: Diagramme zur Materialverteilung

> **Das Muster ist immer dasselbe**: Über alle Stäbe iterieren, Attribute lesen, eine Bedingung prüfen *oder* einen Wert aufsummieren, Ergebnis ausgeben. **Prüfen** und **Auswerten** sind zwei Seiten derselben Medaille.

---

## Setup

Führt diese Zelle einmal aus. Sie lädt die Bibliotheken und das Brückenmodell.

In [ ]:
# Setup – einmal ausführen, dann nicht mehr anfassen
import json
import plotly.graph_objects as go

# Brücke 2 (erweitert) – als JSON eingebettet
bruecke_json = r'''{
  "meta": {
    "projekt": "Velobrücke Aaretal",
    "variante": "Brücke 2 - erweitert (W10)",
    "version": "2.0",
    "beschreibung": "3-Feld-Brücke (10m/12m/10m), Stahl-Holz-Hybrid. Erweitert um Material und Querschnittsfläche pro Stab."
  },
  "felder": [
    {
      "id": "F1",
      "von_x": 0.0,
      "bis_x": 10.0,
      "laenge_m": 10.0
    },
    {
      "id": "F2",
      "von_x": 10.0,
      "bis_x": 22.0,
      "laenge_m": 12.0
    },
    {
      "id": "F3",
      "von_x": 22.0,
      "bis_x": 32.0,
      "laenge_m": 10.0
    }
  ],
  "knoten": [
    {
      "id": "K01",
      "x": 0.0,
      "y": 2.0,
      "z": 0.0
    },
    {
      "id": "K02",
      "x": 10.0,
      "y": 2.0,
      "z": 0.0
    },
    {
      "id": "K03",
      "x": 22.0,
      "y": 2.0,
      "z": 0.0
    },
    {
      "id": "K04",
      "x": 32.0,
      "y": 2.0,
      "z": 0.0
    },
    {
      "id": "K05",
      "x": 0.0,
      "y": 2.0,
      "z": 5.0
    },
    {
      "id": "K06",
      "x": 10.0,
      "y": 2.0,
      "z": 5.0
    },
    {
      "id": "K07",
      "x": 22.0,
      "y": 2.0,
      "z": 5.0
    },
    {
      "id": "K08",
      "x": 32.0,
      "y": 2.0,
      "z": 5.0
    },
    {
      "id": "K09",
      "x": 0.0,
      "y": 0.0,
      "z": 5.0
    },
    {
      "id": "K10",
      "x": 10.0,
      "y": 0.0,
      "z": 5.0
    },
    {
      "id": "K11",
      "x": 22.0,
      "y": 0.0,
      "z": 5.0
    },
    {
      "id": "K12",
      "x": 32.0,
      "y": 0.0,
      "z": 5.0
    },
    {
      "id": "K13",
      "x": 16.0,
      "y": 2.0,
      "z": 5.0
    }
  ],
  "staebe": [
    {
      "id": "S01",
      "von": "K01",
      "bis": "K05",
      "typ": "Stütze",
      "profil": "HEB 300",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0149,
      "laenge_m": 5.0
    },
    {
      "id": "S02",
      "von": "K02",
      "bis": "K06",
      "typ": "Stütze",
      "profil": "HEB 300",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0149,
      "laenge_m": 5.0
    },
    {
      "id": "S03",
      "von": "K03",
      "bis": "K07",
      "typ": "Stütze",
      "profil": "HEB 300",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0149,
      "laenge_m": 5.0
    },
    {
      "id": "S04",
      "von": "K04",
      "bis": "K08",
      "typ": "Stütze",
      "profil": "HEB 300",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0149,
      "laenge_m": 5.0
    },
    {
      "id": "HT01",
      "von": "K05",
      "bis": "K06",
      "typ": "Hauptträger",
      "profil": "HEB 340",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0171,
      "laenge_m": 10.0
    },
    {
      "id": "HT02",
      "von": "K06",
      "bis": "K13",
      "typ": "Hauptträger",
      "profil": "HEB 340",
      "material": "Aluminium",
      "querschnittsflaeche_m2": 0.0171,
      "laenge_m": 6.0
    },
    {
      "id": "HT03",
      "von": "K13",
      "bis": "K07",
      "typ": "Hauptträger",
      "profil": "HEB 340",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0171,
      "laenge_m": 6.0
    },
    {
      "id": "HT04",
      "von": "K07",
      "bis": "K08",
      "typ": "Hauptträger",
      "profil": "HEB 340",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0171,
      "laenge_m": 10.0
    },
    {
      "id": "QT01",
      "von": "K05",
      "bis": "K09",
      "typ": "Querträger",
      "profil": "HEB 200",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0078,
      "laenge_m": 2.0
    },
    {
      "id": "QT02",
      "von": "K06",
      "bis": "K10",
      "typ": "Querträger",
      "profil": "HEB 200",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0078,
      "laenge_m": 2.0
    },
    {
      "id": "QT03",
      "von": "K08",
      "bis": "K12",
      "typ": "Querträger",
      "profil": "HEB 200",
      "material": "Stahl S355",
      "querschnittsflaeche_m2": 0.0078,
      "laenge_m": 2.0
    },
    {
      "id": "GL01",
      "von": "K09",
      "bis": "K10",
      "typ": "Geländer",
      "profil": "BSH 200x200",
      "material": "Holz BSH GL28h",
      "querschnittsflaeche_m2": 0.04,
      "laenge_m": 10.0
    },
    {
      "id": "GL02",
      "von": "K10",
      "bis": "K11",
      "typ": "Geländer",
      "profil": "BSH 200x200",
      "material": "Holz BSH GL28h",
      "querschnittsflaeche_m2": null,
      "laenge_m": 12.0
    },
    {
      "id": "GL03",
      "von": "K11",
      "bis": "K12",
      "typ": "Geländer",
      "profil": "BSH 200x200",
      "material": "Holz BSH GL28h",
      "querschnittsflaeche_m2": 0.04,
      "laenge_m": 10.0
    }
  ],
  "material_stammdaten": {
    "Stahl S355": {
      "dichte_kg_m3": 7850,
      "kategorie": "Metall"
    },
    "Holz BSH GL28h": {
      "dichte_kg_m3": 460,
      "kategorie": "Holz"
    }
  },
  "erlaubte_materialien": [
    "Stahl S355",
    "Holz BSH GL28h"
  ]
}'''

bruecke = json.loads(bruecke_json)

print(f"Projekt:  {bruecke['meta']['projekt']}")
print(f"Variante: {bruecke['meta']['variante']}")
print(f"Felder:   {len(bruecke['felder'])}")
print(f"Knoten:   {len(bruecke['knoten'])}")
print(f"Stäbe:    {len(bruecke['staebe'])}")
print()
print("Erlaubte Materialien:", bruecke["erlaubte_materialien"])


---
## Recap Woche 8

In W8 habt ihr **eingebaute Fehler** gefunden (falsches Profil, negative Höhe, verwaister Knoten).

Die Brücke heute hat eine andere Aufgabe: das geometrische Modell ist **sauber**, aber pro Stab wurden **neue Attribute** ergänzt. Schaut euch zuerst die ersten Stäbe an:

In [ ]:
# Die ersten 3 Stäbe – was ist neu?
for stab in bruecke["staebe"][:3]:
    print(stab)
    print()


**Neu pro Stab**:
- `material` – z.B. `"Stahl S355"` oder `"Holz BSH GL28h"`
- `querschnittsflaeche_m2` – Profilfläche in m² (z.B. `0.0171` für HEB 340)
- `laenge_m` – aus W8 bereits berechnet, jetzt direkt in der Datei

Mit diesen drei Werten könnt ihr **Volumen** ($V = A \cdot L$) und **Masse** ($M = V \cdot \rho$) berechnen.

---

## Konzept: Prüfen und Auswerten – dasselbe Muster

Schaut diese zwei Code-Skelette an. Was ist *strukturell* gleich, was unterscheidet sich?

```python
# Prüfen                              # Auswerten
fehler = []                           summe = 0
for stab in bruecke["staebe"]:        for stab in bruecke["staebe"]:
    if stab["material"] not in OK:        summe += stab["laenge_m"]
        fehler.append(stab["id"])     print(summe)
print(fehler)
```

| Schritt              | Prüfen                  | Auswerten            |
|----------------------|-------------------------|----------------------|
| 1. Iterieren         | `for stab in ...`       | `for stab in ...`    |
| 2. Attribut lesen    | `stab["material"]`      | `stab["laenge_m"]`   |
| 3. Vergleich / Sammeln | `if ... not in ...`   | `summe += ...`       |
| 4. Ergebnis          | Liste der Fehler        | Eine Zahl            |

Wenn ihr Aufgabe 1 versteht, versteht ihr alle Aufgaben heute.

---

# Teil A – Modellprüfung erweitert

Wir prüfen die **neuen Attribute**. Zwei Regeln:
- **Regel 1**: `material` muss in der erlaubten Liste sein.
- **Regel 2**: `querschnittsflaeche_m2` muss vorhanden (nicht `None`) und > 0 sein.

Im Modell sind **2 absichtliche Fehler** versteckt. Findet sie.

### ✏️ Aufgabe 1: Welche Materialien kommen vor?

Listet alle **einzigartigen** Materialien auf, die im Modell vorkommen. Wenn ein nicht-erlaubtes Material in der Liste auftaucht, habt ihr Regel 1 schon halb geprüft.

*Hinweis*: `set()` entfernt Duplikate aus einer Liste.

In [ ]:
# Aufgabe 1: Einzigartige Materialien sammeln
# Worked example - nur 1 Lücke

materialien = []

for stab in bruecke["staebe"]:
    materialien.append(stab["___"])    # ← Welches Attribut?

einzigartige_materialien = set(materialien)

print("Im Modell verwendete Materialien:")
for m in einzigartige_materialien:
    print(f"  - {m}")


### ✏️ Aufgabe 2: Regel 1 – Material erlaubt?

Geht alle Stäbe durch und sammelt diejenigen, deren Material **nicht** in der erlaubten Liste steht.

*Tipp*: `bruecke["erlaubte_materialien"]` enthält die Liste.

In [ ]:
# Aufgabe 2: Regel 1 - Material in erlaubter Liste?

erlaubt = bruecke["erlaubte_materialien"]
fehler_material = []

for stab in bruecke["staebe"]:
    if stab["material"] not in ___:           # ← worauf prüfen?
        fehler_material.append({
            "id":       stab["id"],
            "material": stab["___"],          # ← welches Attribut soll mit?
        })

print(f"Stäbe mit ungültigem Material: {len(fehler_material)}")
for f in fehler_material:
    print(f"  ❌ {f['id']}: Material = '{f['material']}'")


### ✏️ Aufgabe 3: Regel 2 – Querschnittsfläche vorhanden?

Sammelt Stäbe, bei denen `querschnittsflaeche_m2` **fehlt** (`None`) oder **nicht positiv** ist.

In [ ]:
# Aufgabe 3: Regel 2 - Querschnittsfläche valide?

fehler_querschnitt = []

for stab in bruecke["staebe"]:
    A = stab["___"]                                 # ← Attributname
    if A is None or A <= ___:                       # ← was ist die Untergrenze?
        fehler_querschnitt.append(stab["___"])      # ← was speichert ihr?

print(f"Stäbe ohne gültigen Querschnitt: {len(fehler_querschnitt)}")
for sid in fehler_querschnitt:
    print(f"  ❌ {sid}")


### 🎨 3D-Visualisierung der Prüfung

Diese Zelle ist **fertig**. Sie zeigt eure Brücke in 3D – grün = OK, rot = Fehler. Die Funktion nutzt eure Listen `fehler_material` und `fehler_querschnitt`.

In [ ]:
# 3D-Visualisierung: rot = Fehler, grün = OK
# Zelle einfach ausführen.

knoten_lookup = {k["id"]: k for k in bruecke["knoten"]}
fehler_ids = {f["id"] for f in fehler_material} | set(fehler_querschnitt)

fig = go.Figure()

# Stäbe
for stab in bruecke["staebe"]:
    a = knoten_lookup[stab["von"]]
    b = knoten_lookup[stab["bis"]]
    farbe = "#E74C3C" if stab["id"] in fehler_ids else "#27AE60"  # rot/grün
    breite = 8 if stab["id"] in fehler_ids else 4
    fig.add_trace(go.Scatter3d(
        x=[a["x"], b["x"]], y=[a["y"], b["y"]], z=[a["z"], b["z"]],
        mode="lines+text",
        line=dict(color=farbe, width=breite),
        text=["", stab["id"]], textposition="middle center",
        textfont=dict(size=10, color=farbe),
        name=stab["id"], showlegend=False,
    ))

# Knoten
fig.add_trace(go.Scatter3d(
    x=[k["x"] for k in bruecke["knoten"]],
    y=[k["y"] for k in bruecke["knoten"]],
    z=[k["z"] for k in bruecke["knoten"]],
    mode="markers+text",
    marker=dict(size=4, color="#34495E"),
    text=[k["id"] for k in bruecke["knoten"]],
    textposition="top center",
    textfont=dict(size=9, color="#34495E"),
    name="Knoten", showlegend=False,
))

fig.update_layout(
    title=f"Brücke 2 (erweitert) – Prüfung: {len(fehler_ids)} Fehler",
    scene=dict(xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]",
               aspectmode="data"),
    height=550, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

print(f"Identifizierte Fehler-Stäbe: {sorted(fehler_ids)}")


**Zwischenstand**: Ihr habt die fehlerhaften Stäbe identifiziert. Bei der Mengenermittlung in Teil B müsst ihr diese Stäbe ausschliessen.

---

# Teil B – Mengen und Auswertungen

Jetzt extrahiert ihr **Werte** aus dem Modell. Das gleiche `for`-Skelett wie in Teil A, aber statt `if` summiert ihr.

> **Wichtig**: Ihr arbeitet auf den **gültigen** Stäben. Stäbe mit Fehlern (HT02 und GL02) werden ausgeschlossen.

In [ ]:
# Hilfslogik: Liste der GÜLTIGEN Stäbe (vorbereitet, einfach ausführen)

gueltige_staebe = []
for stab in bruecke["staebe"]:
    A = stab["querschnittsflaeche_m2"]
    if stab["material"] in bruecke["erlaubte_materialien"] and A is not None and A > 0:
        gueltige_staebe.append(stab)

print(f"Gültige Stäbe: {len(gueltige_staebe)} von {len(bruecke['staebe'])}")


### ✏️ Aufgabe 4: Anzahl Stäbe pro Material

Zählt, wie viele Stäbe je Material vorkommen. Das Ergebnis ist ein Dictionary `{material: anzahl}`.

*Tipp*: `dict.get(key, 0)` gibt 0 zurück, wenn der Schlüssel noch nicht existiert.

In [ ]:
# Aufgabe 4: Anzahl pro Material

anzahl_pro_material = {}

for stab in gueltige_staebe:
    m = stab["material"]
    anzahl_pro_material[m] = anzahl_pro_material.get(m, 0) + ___    # ← um wie viel hochzählen?

print("Anzahl Stäbe pro Material:")
for material, n in anzahl_pro_material.items():
    print(f"  {material:<25} {n} Stäbe")


### ✏️ Aufgabe 5: Gesamtlänge pro Material

Summiert die Länge `laenge_m` aller gültigen Stäbe pro Material auf.

In [ ]:
# Aufgabe 5: Gesamtlänge pro Material

laenge_pro_material = {}

for stab in gueltige_staebe:
    m = stab["___"]                                          # ← welches Attribut gruppiert?
    L = stab["___"]                                          # ← welches Attribut wird summiert?
    laenge_pro_material[m] = laenge_pro_material.get(m, 0) + ___  # ← was wird addiert?

print("Gesamtlänge pro Material:")
for material, L in laenge_pro_material.items():
    print(f"  {material:<25} {L:6.2f} m")


### ✏️ Aufgabe 6: Volumen und Masse pro Material

Berechnet pro Material:
- **Volumen** $V = A \cdot L$ (Querschnittsfläche mal Länge, je Stab, dann aufsummiert)
- **Masse** $M = V \cdot \rho$ (Dichte aus den Materialstammdaten)

Die Stammdaten findet ihr unter `bruecke["material_stammdaten"]`.

In [ ]:
# Aufgabe 6: Volumen und Masse pro Material

stammdaten = bruecke["material_stammdaten"]

volumen_pro_material = {}
masse_pro_material   = {}

for stab in gueltige_staebe:
    m = stab["material"]
    L = stab["laenge_m"]
    A = stab["querschnittsflaeche_m2"]

    V_stab = ___ * ___                                       # ← Volumen-Formel V = A * L

    rho = stammdaten[m]["dichte_kg_m3"]
    M_stab = V_stab * ___                                    # ← Masse-Formel M = V * ρ

    volumen_pro_material[m] = volumen_pro_material.get(m, 0) + V_stab
    masse_pro_material[m]   = masse_pro_material.get(m, 0) + M_stab

print(f"{'Material':<25} {'V [m³]':>10} {'Masse [kg]':>12}")
print("-" * 50)
for material in volumen_pro_material:
    V = volumen_pro_material[material]
    M = masse_pro_material[material]
    print(f"{material:<25} {V:>10.4f} {M:>12.1f}")

gesamtmasse = sum(masse_pro_material.values())
print("-" * 50)
print(f"{'Gesamt':<25} {'':>10} {gesamtmasse:>12.1f}")
print(f"\nProbe: {gesamtmasse/32:.1f} kg pro Meter Brücke (32 m gesamt)")


**Plausibilität**: Eine Velobrücke wiegt typischerweise 150–300 kg pro Meter. Liegt euer Wert in dem Bereich? Falls nicht, ist irgendwo ein Fehler drin.

---

# Teil C – Auswertungen visualisieren

Tabellen sind nüchtern. Diagramme erzählen sofort die Geschichte. Wir nutzen Plotly (kennt ihr aus W8).

### ✏️ Aufgabe 7: Bar-Chart – Länge pro Material

Erstellt ein Balkendiagramm: x-Achse = Material, y-Achse = Gesamtlänge.

*Tipp*: `list(d.keys())` und `list(d.values())` aus einem Dictionary.

In [ ]:
# Aufgabe 7: Bar-Chart Länge pro Material

materialien = list(laenge_pro_material.___())     # ← keys oder values?
laengen     = list(laenge_pro_material.___())     # ← keys oder values?

fig_bar = go.Figure(data=[
    go.Bar(x=materialien, y=laengen, marker_color=["#2C3E50", "#A0522D"])
])
fig_bar.update_layout(
    title="Gesamtlänge pro Material",
    xaxis_title="Material",
    yaxis_title="Länge [m]",
    height=400,
)
fig_bar.show()


### ✏️ Aufgabe 8: Pie-Chart – Massenanteil pro Material

Erstellt ein Tortendiagramm der **Massenverteilung** (welcher Anteil der Gesamtmasse ist Stahl, welcher Holz?).

In [ ]:
# Aufgabe 8: Pie-Chart Massenanteil

material_labels = list(masse_pro_material.keys())
massen          = list(masse_pro_material.___())     # ← keys oder values?

fig_pie = go.Figure(data=[
    go.Pie(labels=material_labels, values=massen,
           marker_colors=["#2C3E50", "#A0522D"],
           textinfo="label+percent")
])
fig_pie.update_layout(
    title="Massenanteil pro Material",
    height=400,
)
fig_pie.show()


---
## 🎯 Heureka – Bericht

Setzt alle Erkenntnisse in einem kompakten **Bericht** zusammen. Diese Zelle ist fertig vorbereitet.

In [ ]:
# Heureka: Kompakter Auswertungsbericht (Zelle einfach ausführen)

print("=" * 60)
print(f"AUSWERTUNGSBERICHT – {bruecke['meta']['variante']}")
print("=" * 60)

# Modell-Übersicht
print(f"\n📐 MODELL")
print(f"   Brückenlänge:    {sum(f['laenge_m'] for f in bruecke['felder'])} m ({len(bruecke['felder'])} Felder)")
print(f"   Knoten:          {len(bruecke['knoten'])}")
print(f"   Stäbe gesamt:    {len(bruecke['staebe'])}")

# Prüfung
print(f"\n🔍 MODELLPRÜFUNG")
print(f"   Fehler Material:       {len(fehler_material)}")
print(f"   Fehler Querschnitt:    {len(fehler_querschnitt)}")
print(f"   Gültige Stäbe:         {len(gueltige_staebe)} / {len(bruecke['staebe'])}")

# Mengen
print(f"\n📦 MENGEN (gültige Stäbe)")
print(f"   {'Material':<20} {'Anzahl':>7} {'Länge [m]':>11} {'V [m³]':>9} {'M [kg]':>9}")
print(f"   {'-'*20} {'-'*7} {'-'*11} {'-'*9} {'-'*9}")
for m in volumen_pro_material:
    print(f"   {m:<20} {anzahl_pro_material[m]:>7} {laenge_pro_material[m]:>11.2f} "
          f"{volumen_pro_material[m]:>9.4f} {masse_pro_material[m]:>9.1f}")

print(f"\n   Gesamtmasse: {gesamtmasse:.1f} kg = {gesamtmasse/1000:.2f} t")
print(f"   Pro Meter:   {gesamtmasse/32:.1f} kg/m")
print("\n" + "=" * 60)


---
## 🚀 Ausblick: Eure Projektarbeit (ab Woche 11)

Was ihr heute gemacht habt, ist die **Mini-Version eurer Projektarbeit**:

| Heute (W10)                               | Projektarbeit (ab W11)                   |
|-------------------------------------------|-------------------------------------------|
| Bestehende JSON-Daten laden               | Eigenes oder vorgegebenes Modell laden   |
| 2 Prüfregeln auf Material/Querschnitt     | Eigene Prüfregeln definieren             |
| Mengen pro Material aufsummieren          | Mengen je nach Fragestellung extrahieren |
| Bericht in der Konsole                    | Bericht als Notebook + Präsentation      |

**Das `for`-Skelett bleibt gleich** – nur die Frage ändert sich.

### Selbststudium
- Schaut in das **Demo-Notebook IFC**: gleiche Logik auf einem echten BIM-Modell
- Notiert euch eine Frage dazu...

---
*Digitales Bauen 1 | BFH AHB | Louis Trümpler*